# 27 – Đánh giá Random Forest bốn lớp đã đóng băng (M2-R6b/R6c/R7)

## 1. Giới thiệu mô hình đã đóng băng

`svmg-rf-four-class-m2r6a-0.1.0`: 15 cây, độ sâu 4, `min_samples_leaf=5`, `max_features=sqrt`, `random_state=20261124`, 23 đặc trưng,
bốn lớp, UNKNOWN khi độ tin cậy < τ = 0,5. Dữ liệu `SIMULATION`, nhãn proxy; **không** phải độ chính xác phát hiện nguy hiểm ngoài
thực tế.

Hai loại số liệu được tách rõ:
- **Tính lại trong notebook:** Validation và S0b, bằng mô hình đóng băng (dự đoán thật, không hard-code).
- **Lịch sử, chỉ đọc:** Test-G1 đã đánh giá **đúng một lần** ở M2-R6c. Notebook không nạp Test-G1 và không chạy lại script đánh giá;
  nó đọc `ml/reports/ml/m2_r6c_test_g1_metrics_summary.json` (commit, trích từ `results.json` đã khóa).

Cần dataset v0.3 và mô hình (gitignored); nếu thiếu, chạy từ thư mục gốc repo:

```bash
.venv/bin/python -m ml.train.four_class_experimental_dataset && .venv/bin/python -m ml.train.four_class_v03_dataset
.venv/bin/python -m ml.train.rf_four_class_m2r6
.venv/bin/python -m nbconvert --to notebook --execute ml/notebooks/27_four_class_rf_evaluation.ipynb --output-dir ml/data/models/rf_four_class_m2r6/notebook_runs
```

## 2. Thông tin model và dataset SHA

In [ ]:
import hashlib, json, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.eval import final_test_g1_m2r6c as T
from ml.export import rf4_freeze_m2r6b as F
from ml.export import rf_c_export as E
from ml.train import rf_four_class_m2r6 as R
from ml.train.g1_simulation_dataset import _sha

FIG = R.OUT / "notebook_runs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 180)
csv, dataset_sha = R.DATASETS["v0.3"]
if not csv.exists() or not (F.MODEL_DIR / "model.joblib").exists():
    raise FileNotFoundError("Thiếu dataset v0.3 hoặc mô hình (gitignored): chạy các lệnh ở mục 1 từ thư mục gốc repo.")
manifest = json.loads(F.MANIFEST.read_text())
model, meta = F.load_frozen()          # dừng nếu SHA model.joblib, schema hoặc preprocessing khác bản đóng băng
tau = manifest["unknown_policy"]["tau_conf"]
header = E.generate_model_h(model, R.FEATURES, F.c_metadata(meta), tau)[0]
info = {"model_version": meta["model_version"], "model_sha256 khớp": _sha(F.MODEL_DIR / "model.joblib") == manifest["model_sha256"],
        "dataset_version": meta["dataset_version"], "dataset_sha256 khớp": _sha(csv) == dataset_sha == manifest["dataset_sha256"],
        "header C99 sinh lại khớp": hashlib.sha256(header.encode()).hexdigest() == manifest["c_artifact"]["sha256"],
        "header_sha256": manifest["c_artifact"]["sha256"][:16] + "…", "feature_schema": meta["feature_schema_version"],
        "preprocessing": meta["preprocessing_version"], "lớp": meta["class_mapping"], "siêu tham số": meta["params"], "tau": tau}
display(pd.Series(info, name="giá trị").to_frame())
assert info["model_sha256 khớp"] and info["dataset_sha256 khớp"] and info["header C99 sinh lại khớp"] and tau == 0.5

## 3. Kết quả Validation (tính lại)

CORE Validation: 469 cửa sổ, 12 session; Test-G1 bị loại ngay khi đọc dữ liệu. So với số đã công bố ở M2-R6a.

In [ ]:
frame = R.load(csv)
val, s0b = frame[frame.subset.eq("CORE") & frame.split.eq(R.VAL)], frame[frame.subset.eq("CHALLENGE_S0B")]
p_val, p_s0b = model.predict_proba(R.matrix(val)), model.predict_proba(R.matrix(s0b))
y_val = val.y.to_numpy()
val_before, val_after = R.metrics(y_val, p_val.argmax(1)), T.with_tau(y_val, T.apply_tau(p_val, tau))
display(pd.DataFrame({"trước tau": {k: val_before[k] for k in ("n", "accuracy", "macro_f1", "weighted_f1")},
                      "sau tau = 0,5 (UNKNOWN = bỏ sót)": {k: val_after[k] for k in ("n", "accuracy", "macro_f1", "weighted_f1")}}))
assert abs(val_before["macro_f1"] - 0.9876) <= 1e-4 and val_after["coverage"] == manifest["policy_table"]["0.5"]["validation"]["coverage"]

## 4. Chính sách UNKNOWN τ = 0,5 và 5. Coverage, effective recall

Độ tin cậy = xác suất lá trung bình lớn nhất (**chưa hiệu chuẩn**). Quy tắc khai báo trước ở M2-R6b: chọn τ lớn nhất mà mọi lớp còn recall
hiệu dụng ≥ 0,95 trên Validation. **Không τ nào đạt** (kể cả 0,5: một cửa sổ AMPLITUDE_GROWTH có conf < 0,5), nên giữ giá trị dự phòng
0,5 = mặc định `SVMG_TAU_CONF` của firmware. UNKNOWN luôn tính là bỏ sót, không bao giờ thành NORMAL. Bảng dưới tính lại cả lưới τ
và đối chiếu với manifest đóng băng.

In [ ]:
grid = {str(t): {"validation": F.policy(p_val, y_val, t), "s0b": F.policy(p_s0b, None, t)} for t in F.TAU_GRID}
print("Khớp bảng chính sách trong manifest đóng băng:", grid == manifest["policy_table"])
assert grid == manifest["policy_table"]
pol = pd.DataFrame({t: {"coverage Validation": g["validation"]["coverage"], **{f"recall hiệu dụng {c}": v["effective_recall"]
                    for c, v in g["validation"]["per_class"].items()}, "S0b UNKNOWN": g["s0b"]["unknown"],
                    "S0b báo sai bất thường": g["s0b"]["false_non_normal_on_normal"]} for t, g in grid.items()}).T
display(pol)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
pol.filter(like="recall hiệu dụng").assign(coverage=pol["coverage Validation"]).plot(ax=ax[0], marker="o")
ax[0].axhline(0.95, color="grey", ls="--", lw=1); ax[0].set(title="Validation theo tau", xlabel="tau", ylabel="tỉ lệ")
pol[["S0b UNKNOWN", "S0b báo sai bất thường"]].plot.bar(ax=ax[1], rot=0, title="S0b (304 cửa sổ NORMAL) theo tau")
ax[1].set(xlabel="tau", ylabel="cửa sổ")
fig.tight_layout(); fig.savefig(FIG / "27_tau_policy.png", dpi=110); plt.show()

## 6. Kết quả S0b challenge

S0b là các session có nghiêng **tạm thời** (20–90 s); cửa sổ có nhãn chỉ là NORMAL sau khi sự kiện đã ra khỏi lịch sử 180 s. Đây là tập
challenge, không dùng để huấn luyện hay chọn mô hình.

In [ ]:
s0b_pred = T.apply_tau(p_s0b, tau)
names = R.CLASSES + ["UNKNOWN"]
tab = pd.crosstab(s0b.split.values, pd.Categorical([names[i] for i in s0b_pred], categories=names), dropna=False)
display(tab)
ax = tab.T.plot.bar(rot=15, figsize=(7, 3.5), title="S0b sau tau = 0,5: dự đoán theo split")
ax.set(xlabel="", ylabel="cửa sổ"); plt.tight_layout(); plt.savefig(FIG / "27_s0b.png", dpi=110); plt.show()

## 7. Kết quả Test-G1 đã khóa, 8. Confusion matrix Test-G1, 9. Precision/Recall/F1 Test-G1

**Số liệu lịch sử, chỉ đọc.** Nếu máy có `results.json` đã khóa (gitignored), ô dưới chỉ kiểm tra SHA của nó khớp nguồn của tóm tắt.

In [ ]:
summary = json.loads((ROOT / "ml/reports/ml/m2_r6c_test_g1_metrics_summary.json").read_text())
locked = F.MODEL_DIR / "test_g1" / "results.json"
print("Nguồn:", summary["source"], "| SHA", summary["source_sha256"][:16] + "…")
print("results.json cục bộ:", ("có, SHA khớp " + str(_sha(locked) == summary["source_sha256"])) if locked.exists() else "không có (bản clone sạch) – dùng tóm tắt")
assert summary["model_version"] == meta["model_version"] and summary["tau"] == tau
tb, ta = summary["before_tau"], summary["after_tau"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (title, m) in zip(axes, (("trước tau", tb), ("sau tau = 0,5", ta))):
    cm = np.array(m["confusion"])[:4]
    ax.imshow(cm, cmap="Oranges")
    for i, j in np.ndindex(cm.shape):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set(title=f"Test-G1 (lịch sử) {title}", xticks=range(cm.shape[1]), yticks=range(4), xlabel="dự đoán", ylabel="thật")
    ax.set_xticklabels(m["confusion_labels"], rotation=20); ax.set_yticklabels(R.CLASSES)
fig.tight_layout(); fig.savefig(FIG / "27_confusion_test_g1.png", dpi=110); plt.show()
display(pd.DataFrame({(s, k): {c: m["per_class"][c][k] for c in R.CLASSES} for s, m in (("trước tau", tb), ("sau tau", ta))
                      for k in ("precision", "recall", "f1")}))
print(f"UNKNOWN sau tau: {ta['unknown']} ({ta['unknown_per_class']}); coverage {ta['coverage']}; KTC 95% macro-F1 trước tau theo session {tb['macro_f1_session_bootstrap_95']}")

## 10. So sánh Validation và Test-G1

Validation (12 session) đã dùng để chọn cấu hình nên lạc quan; Test-G1 là ước lượng trung thực hơn **trong cùng miền mô phỏng**.

In [ ]:
cmp = pd.DataFrame({"Validation trước tau (tính lại)": val_before, "Validation sau tau (tính lại)": val_after,
                    "Test-G1 trước tau (lịch sử)": tb, "Test-G1 sau tau (lịch sử)": ta}).loc[["accuracy", "macro_f1", "weighted_f1"]].astype(float)
display(cmp)
ax = cmp.T.plot.bar(rot=12, figsize=(10, 3.8), ylim=(0, 1.05), title="Validation và Test-G1")
ax.set(ylabel="giá trị"); plt.tight_layout(); plt.savefig(FIG / "27_validation_vs_test_g1.png", dpi=110); plt.show()

## 11. Phân tích AMPLITUDE_GROWTH false positives

Ca lỗi liên quan AMPLITUDE_GROWTH trên Test-G1 (sau τ), từ tóm tắt lịch sử. Báo nhầm tập trung ở **hai session NORMAL**, nhiều cửa sổ có
độ tin cậy > 0,5, nên τ không loại được chúng.

In [ ]:
gf = pd.DataFrame(summary["growth_failures"])
display(gf.groupby(["session_id", "label", "pred"]).agg(cửa_sổ=("conf", "size"), conf_min=("conf", "min"), conf_max=("conf", "max"),
                                                       độ_phủ_trung_vị=("target_coverage_180", "median")))
fig, ax = plt.subplots(figsize=(9, 3.6))
for (sid, lbl), g in gf.groupby(["session_id", "label"]):
    ax.scatter(g.window_end_seconds, g.conf, label=f"{sid} ({lbl}→{g.pred.iloc[0]})")
ax.axhline(tau, color="grey", ls="--", lw=1, label="tau = 0,5")
ax.set(title="Test-G1 (lịch sử): độ tin cậy các ca lỗi liên quan AMPLITUDE_GROWTH", xlabel="thời điểm cuối cửa sổ (s)", ylabel="conf")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIG / "27_growth_failures.png", dpi=110); plt.show()

## 12. So sánh với Baseline B

Baseline B là luật bốn lớp của Core C (`core/src/baseline.c`, tham số demo mặc định, **không** hiệu chỉnh lại), chạy trên cùng cửa sổ và
cùng 23 đặc trưng. Validation tính lại (biên dịch đúng hàm C qua ctypes); Test-G1 là số lịch sử.

In [ ]:
base_val = R.metrics(y_val, R.baseline_b(R.matrix(val)))
rows = {"RF Validation": val_before, "Baseline B Validation": base_val, "RF Test-G1 (lịch sử)": tb, "Baseline B Test-G1 (lịch sử)": summary["baseline_b"]}
display(pd.DataFrame({k: {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"], **{f"recall {c}": m["per_class"][c]["recall"] for c in R.CLASSES}}
                      for k, m in rows.items()}))

## 13. Kết quả C99/Python parity và 14. Kết quả ESP32 Wokwi shadow

Parity lấy từ manifest đóng băng (Validation, S0b, vector tại ngưỡng) và tóm tắt Test-G1 (lịch sử).

Wokwi (lịch sử, M2-R7/M2-R8, gitignored `ml/data/parity/wokwi_mpu6050/`; chi tiết `ml/docs/M2_handover.md`):

| Lần ghi | Nguồn | Kết quả |
|---|---|---|
| `rf4_sim_01` | SIMULATION (**stub sóng sin của M1**) | 23/23 đặc trưng; RF chạy 28 lần, dự đoán NORMAL; tối đa 719 µs; Baseline B vẫn là mô hình chính |
| `rf4_02` | SENSOR (MPU6050 mô phỏng, đứng yên) | tối đa 20/23 đặc trưng (3 đặc trưng 180 s không xác định khi không chuyển động) → RF không được gọi, `rf=UNKNOWN` |

Chưa có bằng chứng RF chạy trên dữ liệu cảm biến chuyển động, chưa đo trên ESP32 thật.

In [ ]:
par = {k: manifest["parity"][k] for k in ("validation", "s0b", "threshold_boundaries")} | {"test_g1 (lịch sử)": summary["c99_parity"]}
display(pd.DataFrame(par).loc[["vectors", "pred_mismatches", "proba_max_abs_diff", "unknown_mismatches_at_tau", "min_top2_margin"]])
print("Đầu vào lỗi trả -1:", manifest["parity"]["invalid_inputs"])
for run in ("rf4_sim_01", "rf4_02"):
    p = ROOT / f"ml/data/parity/wokwi_mpu6050/{run}/summary.json"
    print(run, json.loads(p.read_text()).get("rf_shadow") if p.exists() else "không có bản ghi cục bộ (gitignored) – xem bảng ở trên")

## 15. Giới hạn và kết luận nghiên cứu

- Mô hình tái lập và đóng băng được (SHA dataset, mô hình, header C99 khớp), C99 khớp Python tuyệt đối về lớp trên mọi tập đã thử.
- Validation (macro-F1 0,988) lạc quan; Test-G1 một lần: macro-F1 hiệu dụng **0,816**, coverage 0,917, precision AMPLITUDE_GROWTH
  **0,34** (báo nhầm tập trung ở hai session NORMAL). Baseline B thấp hơn (Test-G1 macro-F1 0,53) và bỏ sót toàn bộ GROWTH.
- Nhãn là proxy mô phỏng sinh từ ngưỡng trên roll; RF học lại luật đó. Không phải dự báo lật tàu, không phải đánh giá GM/GZ.
- Chưa có Test-G2, dữ liệu MPU6050 thật, kiểm chứng trên tàu hay bằng chứng RF trên cảm biến chuyển động; RF chỉ chạy **shadow**,
  Baseline B và ngưỡng cứng `phi_hard` vẫn quyết định cảnh báo.